# 1 · Del comportamiento al modelo: inferencia

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/decostruttivismo/IIC3800-2026-NCC-IA/blob/main/clase-24-agosto/notebooks/01_inferencia.ipynb)

**Modelos de Aprendizaje en Neurociencia Cognitiva e Inteligencia Artificial**  
Clase del 24 de agosto · Interferencia cognitiva (Stroop)

---

### Qué vamos a hacer

Tenemos los datos (simulados) de un experimento Stroop: 30 participantes × 100 ensayos.
En cada ensayo hay un **tiempo de reacción** y si la respuesta fue **correcta**.

La pregunta de la clase es cómo se pasa de esa tabla a una afirmación como
*«la incongruencia cuesta 65 milisegundos»*, y qué hace falta creer para que esa
afirmación esté justificada.

El esquema que organiza todo es

$$Y = f(X, \theta) + \epsilon$$

| símbolo | en este experimento |
|---|---|
| $X$ | la congruencia del estímulo (y otras variables que midamos) |
| $Y$ | el tiempo de reacción, o el acierto |
| $\theta$ | los parámetros latentes: cuánto pesa cada cosa |
| $\epsilon$ | la variabilidad que el modelo no explica |

Este cuaderno recorre cuatro modelos de $f$, cada uno de los cuales arregla un
problema del anterior.

> **Cómo se ejecuta una celda:** clic en la celda y `Shift + Enter`.  
> Las celdas marcadas **✋ TU TURNO** son para que escribas tú.

## 1. Cargar los datos

El CSV se lee por URL desde el repositorio del curso: no hay que subir nada.

In [ ]:
# --- Configuración: de dónde se leen los datos -------------------------------
USUARIO = "decostruttivismo"
REPO    = "IIC3800-2026-NCC-IA"
RAMA    = "main"
CARPETA = "clase-24-agosto"
ARCHIVO = "datos/clase_24agosto_dataset.csv"

URL = f"https://raw.githubusercontent.com/{USUARIO}/{REPO}/{RAMA}/{CARPETA}/{ARCHIVO}"

import os
CANDIDATOS = [ARCHIVO,
              os.path.join("..", ARCHIVO),
              os.path.join("..", "..", ARCHIVO),
              os.path.basename(ARCHIVO)]
FUENTE = next((p for p in CANDIDATOS if os.path.exists(p)), URL)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv(FUENTE)
print("Leyendo desde:", FUENTE)
print("Filas y columnas:", data.shape)
data.head()

Las columnas son:

| columna | qué es |
|---|---|
| `subject` | identificador del participante (`S01`…`S30`) |
| `trial` | número de ensayo dentro del participante (1–100) |
| `condition` | `Congruent` o `Incongruent` |
| `RT_ms` | tiempo de reacción en milisegundos |
| `accuracy` | 1 = correcto, 0 = incorrecto |
| `age` | edad del participante en años |

Antes de modelar, conviene mirar la forma de los datos: cuántos participantes hay,
si el diseño está balanceado y si falta algo.

In [ ]:
print('Participantes:', data['subject'].nunique())
print('Ensayos por participante:', data.groupby('subject').size().unique())
print()
print(data['condition'].value_counts())
print()
print('Valores faltantes por columna:')
print(data.isna().sum())

## 2. La primera mirada: promedios por condición

El resumen más simple posible de un experimento de dos condiciones.

In [ ]:
data.groupby('condition')[['RT_ms', 'accuracy']].mean()

Miremos la distribución completa antes de resumirla.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

for cond, color in [('Congruent', '#4C72B0'), ('Incongruent', '#C44E52')]:
    ax[0].hist(data.loc[data['condition'] == cond, 'RT_ms'],
               bins=40, alpha=0.6, label=cond, color=color)
ax[0].set_xlabel('RT (ms)'); ax[0].set_ylabel('ensayos')
ax[0].set_title('Distribución del RT'); ax[0].legend()

medias = data.groupby('condition')['RT_ms'].mean()
ax[1].bar(medias.index, medias.values, color=['#4C72B0', '#C44E52'])
ax[1].set_ylabel('RT medio (ms)'); ax[1].set_ylim(0, 650)
ax[1].set_title('Medias por condición')

plt.tight_layout(); plt.show()

Las dos distribuciones se solapan mucho. El efecto existe **en promedio**, pero un
ensayo incongruente cualquiera puede ser perfectamente más rápido que un ensayo
congruente cualquiera. Recuerda esta imagen: es exactamente la razón por la que en el
cuaderno 02 la predicción va a resultar mucho más pobre de lo que sugiere el valor-p.

> ### ✋ TU TURNO — 1
>
> Calcula el **efecto Stroop de cada participante**: para cada `subject`, el RT medio
> incongruente menos el RT medio congruente.
>
> Preguntas a contestar mirando el resultado:
>
> - ¿Cuántos de los 30 participantes muestran el efecto en la dirección esperada?
> - ¿Cuál es el rango del efecto entre participantes?
>
> *Pista:* `data.groupby(['subject','condition'])['RT_ms'].mean().unstack()`

In [ ]:
# Escribe aquí tu respuesta.



## 3. Modelo 1 — regresión lineal simple

Escribimos la media como un modelo. Para el ensayo $i$:

$$RT_i = \beta_0 + \beta_1 \cdot \text{Condición}_i + \epsilon_i, \qquad \text{Condición} \in \{0, 1\}$$

con `Congruent` codificado como 0. Entonces $\beta_0$ es el RT congruente medio y
$\beta_1$ es **exactamente** la diferencia entre condiciones: el efecto de interferencia.

La ganancia respecto de `groupby` no está en el número, que es el mismo, sino en que
ahora tenemos un error estándar, un intervalo de confianza y un valor-p.

In [ ]:
import statsmodels.formula.api as smf

m1 = smf.ols('RT_ms ~ C(condition)', data=data).fit()
print(m1.summary())

### Cómo leer esa salida

| en el `summary()` | valor | qué significa |
|---|---|---|
| `Intercept` | 518.6 | RT medio en congruente ($\beta_0$) |
| `C(condition)[T.Incongruent]` | +65.06 | el efecto Stroop en ms ($\beta_1$) |
| `std err` | 2.79 | incertidumbre sobre ese 65.06 |
| `P>|t|` | ~1e-110 | probabilidad de ver algo así de extremo si $\beta_1$ fuera 0 |
| `R-squared` | 0.154 | fracción de la varianza del RT que el modelo explica |

Las dos últimas filas dicen cosas muy distintas y conviene no confundirlas:

- El **valor-p** es astronómicamente pequeño. El efecto es todo lo fiable que se puede pedir.
- El **R²** es 0.15. El modelo explica el 15 % de la variación del RT; el 85 % restante
  es otra cosa.

Un efecto puede ser certísimo y minúsculo a la vez. Esta es la distinción que el
cuaderno 02 lleva hasta el final.

## 4. Modelo 2 — añadir el número de ensayo

Un experimento de 100 ensayos no es 100 repeticiones intercambiables: la gente practica,
y también se cansa. Si el RT deriva a lo largo de la sesión y esa deriva no está en el
modelo, se va a $\epsilon$ e infla la incertidumbre.

$$RT_i = \beta_0 + \beta_1 \cdot \text{Condición}_i + \beta_2 \cdot \text{Ensayo}_i + \epsilon_i$$

In [ ]:
m2 = smf.ols('RT_ms ~ C(condition) + trial', data=data).fit()
print(m2.summary().tables[1])
print('R² =', round(m2.rsquared, 4))

El coeficiente de `trial` es **−0.549 ms por ensayo**: a lo largo de los 100 ensayos
los participantes se aceleran unos 55 ms. Es práctica, no efecto Stroop, y el R² sube
de 0.154 a 0.190 solo por haberla nombrado.

Nota que $\beta_1$ casi no se mueve (65.06 → 64.67). Eso es lo esperable en un diseño
**aleatorizado**: como la condición se asignó al azar a lo largo de la sesión, no está
correlacionada con `trial`, y añadir `trial` no puede sesgar la estimación del efecto —
solo la hace más precisa. En datos observacionales esto no se cumple, y ahí añadir una
variable sí cambia las conclusiones.

> ### ✋ TU TURNO — 2
>
> Dos cosas:
>
> 1. ¿El efecto Stroop **cambia** a lo largo de la sesión? Ajusta un modelo con
>    interacción: `'RT_ms ~ C(condition) * trial'` y mira el término de interacción.
> 2. Añade `age` al modelo 2. ¿La edad predice el RT en esta muestra?
>
> En ambos casos, decide qué concluyes mirando el coeficiente **y** su valor-p.

In [ ]:
# Escribe aquí tu respuesta.



## 5. El problema: los ensayos no son independientes

La regresión de arriba supone que las 3000 filas son 3000 observaciones independientes.
No lo son: vienen de 30 personas, 100 filas cada una. Si un participante es lento, sus
100 filas son lentas juntas.

Cuánto importa esto es una cuestión empírica. Miremos.

In [ ]:
por_sujeto = data.groupby('subject')['RT_ms'].mean().sort_values()

por_sujeto.plot(kind='bar', figsize=(11, 3), color='#55A868')
plt.axhline(data['RT_ms'].mean(), color='k', ls='--', lw=1, label='media global')
plt.ylabel('RT medio (ms)'); plt.xlabel('')
plt.title('RT medio por participante'); plt.legend(); plt.tight_layout(); plt.show()

print('Del más rápido al más lento: %.1f a %.1f ms' % (por_sujeto.min(), por_sujeto.max()))

El participante más rápido promedia 434 ms y el más lento 665 ms: una diferencia de
**231 ms**, más de tres veces el efecto que nos interesa medir.

Esa variabilidad entre personas no es ruido experimental: es sistemática, y es el
término que aísla la descomposición de la lámina,

$$\underbrace{\text{Observado}}_{RT_{ij}} = \underbrace{\text{Señal del experimento}}_{\text{efecto fijo: condición, ensayo}} + \underbrace{\text{Señal individual}}_{\text{efecto aleatorio: } u_j} + \underbrace{\text{Ruido}}_{\epsilon_{ij}}$$

El modelo mixto añade un intercepto propio por participante:

$$RT_{ij} = \beta_0 + \beta_1 \text{Condición}_{ij} + \beta_2 \text{Ensayo}_{ij} + u_j + \epsilon_{ij}, \qquad u_j \sim \mathcal{N}(0, \sigma^2_{\text{sujeto}})$$

El subíndice $j$ es el participante, $i$ el ensayo. $u_j$ dice «este participante es
$u_j$ milisegundos más lento que la media», y se estima como una desviación con
distribución, no como 30 parámetros libres.

In [ ]:
m3 = smf.mixedlm('RT_ms ~ C(condition) + trial',
                 data=data,
                 groups=data['subject']).fit()
print(m3.summary())

In [ ]:
var_sujeto = float(m3.cov_re.iloc[0, 0])
var_resid  = float(m3.scale)
icc = var_sujeto / (var_sujeto + var_resid)

print('Varianza entre participantes : %8.1f' % var_sujeto)
print('Varianza residual            : %8.1f' % var_resid)
print('ICC                          : %8.3f' % icc)

### Qué cambió

| | OLS (m2) | mixto (m3) |
|---|---|---|
| efecto de condición | 64.67 ms | 64.67 ms |
| error estándar | 2.73 | **1.69** |

La estimación es idéntica; la **incertidumbre cae un 38 %**. Al sacar la variabilidad
entre personas del término de error, el efecto de condición se mide contra un fondo
mucho más limpio. El modelo mixto no cambió la respuesta, cambió cuánto podemos
confiar en ella.

El **ICC** (coeficiente de correlación intraclase) es 0.625: el 62 % de la varianza del
RT que no explican condición ni ensayo es **variación estable entre personas**. Solo el
38 % restante es variación ensayo a ensayo.

Guarda ese número. En el cuaderno 02 vamos a intentar predecir el RT de participantes
**que el modelo nunca vio**. Ninguno de nuestros predictores —condición, ensayo, edad—
describe a la persona, así que ese 62 % va a quedar fuera del alcance del modelo: no
porque validemos mal, sino porque no hemos medido nada individual.

> ### ✋ TU TURNO — 3
>
> Hasta aquí cada participante tiene su propia *velocidad* (intercepto), pero se le
> impone el *mismo* efecto Stroop. Ajusta un modelo que también deje variar el efecto:
>
> ```python
> m4 = smf.mixedlm('RT_ms ~ C(condition) + trial',
>                  data=data, groups=data['subject'],
>                  re_formula='~C(condition)').fit()
> ```
>
> Compara `Group Var` (variabilidad de la velocidad basal) con
> `C(condition)[T.Incongruent] Var` (variabilidad del efecto). ¿Cuál es mayor, y qué
> significa eso sobre los participantes?
>
> *Puede aparecer un aviso de convergencia; no es un error.*

In [ ]:
# Escribe aquí tu respuesta.



## 6. La otra variable: aciertos

`accuracy` es binaria y la regresión lineal no es la herramienta adecuada, por dos
razones de distinto peso. La conocida es que nada impide a una recta predecir
probabilidades fuera de $[0,1]$ —en estos datos no llega a pasar, porque 0.88 y 0.94
están lejos de los extremos, pero ocurre en cuanto las probabilidades se acercan a 0 o
a 1, o el modelo tiene muchos predictores. La de fondo es que la regresión lineal
supone varianza constante, y en una variable 0/1 la varianza es $p(1-p)$: depende de la
propia media, y es máxima en 0.5 y mínima en los extremos. Esa es la razón por la que
los errores estándar de un OLS sobre una variable binaria no son de fiar aunque las
predicciones caigan dentro del rango.

La regresión logística modela la probabilidad a través de la escala de los **log-odds**:

$$p_{ij} = P(\text{Acierto}_{ij} = 1), \qquad \text{odds}_{ij} = \frac{p_{ij}}{1 - p_{ij}}, \qquad \text{logit}(p_{ij}) = \log \frac{p_{ij}}{1-p_{ij}} = \beta_0 + \beta_1 \text{Condición}_{ij}$$

y equivalentemente

$$p = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X)}}$$

La razón de pasar por el logit es que estira $[0,1]$ a toda la recta real, de modo que
un modelo lineal en esa escala nunca produce una probabilidad imposible.

In [ ]:
modelo_log = smf.logit('accuracy ~ C(condition)', data=data).fit()
print(modelo_log.summary())

Los coeficientes están en log-odds y no se leen directamente. Hay dos traducciones.

In [ ]:
print('Coeficientes (log-odds):')
print(modelo_log.params)
print()

# Traducción 1: a probabilidades
nuevos = pd.DataFrame({'condition': ['Congruent', 'Incongruent']})
nuevos['prob_predicha_correcta'] = modelo_log.predict(nuevos)
print(nuevos)
print()

# Traducción 2: a odds ratios
print('Odds ratios:')
print(np.exp(modelo_log.params))

### Las tres escalas, en el mismo resultado

| escala | congruente | incongruente | efecto |
|---|---|---|---|
| log-odds | 2.752 | 2.031 | $\beta_1 = -0.721$ |
| odds | 15.67 | 7.62 | OR = 0.486 |
| probabilidad | 0.940 | 0.884 | −5.6 puntos |

Cómo se dice cada una en castellano:

- **Odds 15.67** = en congruente hay unas 15.7 respuestas correctas por cada incorrecta.
- **OR 0.486** = pasar a incongruente **reduce a la mitad** los odds de acertar.
- **Probabilidad 0.884** = de cada 100 ensayos incongruentes, unos 88 correctos.

Un aviso que se olvida a menudo: el odds ratio **no** es un cociente de probabilidades.
Los odds se dividen por la mitad, pero la probabilidad solo baja de 0.940 a 0.884. Cuando
las probabilidades son altas, un OR dramático corresponde a un cambio modesto en
probabilidad. Reportar solo el OR es una forma clásica de exagerar un efecto.

Fíjate por último en que las dos medidas **empeoran a la vez**: la incongruencia hace a
la gente más lenta *y* menos precisa. No hay compensación entre velocidad y precisión,
así que el efecto en RT no se puede explicar diciendo que los participantes se tomaron
más tiempo para ser más exactos.

> ### ✋ TU TURNO — 4
>
> 1. Añade `trial` al modelo logístico y comprueba si la precisión también mejora con
>    la práctica. Traduce el efecto a puntos porcentuales y compáralo con los 5.6
>    puntos que cuesta la incongruencia: ¿es `trial` una molestia menor o un efecto
>    del mismo orden que el que estudiamos?
> 2. Calcula a mano la probabilidad de acierto en incongruente a partir de los
>    coeficientes, con $p = 1/(1 + e^{-(\beta_0 + \beta_1)})$, y verifica que coincide
>    con lo que devolvió `.predict()`.

In [ ]:
# Escribe aquí tu respuesta.



---

## Resumen del cuaderno 1

| paso | qué añadió | qué costó |
|---|---|---|
| medias por condición | el efecto existe: 65 ms | ninguna medida de incertidumbre |
| OLS `~ C(condition)` | error estándar, p, R² = 0.15 | supone ensayos independientes |
| OLS `+ trial` | la práctica vale −0.55 ms/ensayo | sigue suponiendo independencia |
| mixto (`mixedlm`, `groups=subject`) | error estándar 38 % menor; ICC = 0.62 | — |
| logístico | efecto en aciertos: OR = 0.49 | los odds no son probabilidades |

Y los dos números que hay que llevarse a la segunda mitad de la clase:

- El efecto de condición tiene $p \approx 10^{-110}$.
- El modelo explica el **15 %** de la varianza del RT.

Todo lo anterior es **inferencia**: preguntamos si un efecto es compatible con el azar
bajo un modelo dado. La otra pregunta —en qué medida el modelo generaliza a
observaciones nuevas— es **predicción**, y no se responde con nada de lo que hicimos aquí.

→ Continúa en **`02_prediccion.ipynb`**.